# XGBoost Hyperparameter Tuning

**Chapter 5: Advanced Classification Methods - EXTRA**

## Learning Objectives

- Master XGBoost hyperparameter tuning
- Use GridSearchCV and RandomizedSearchCV
- Implement early stopping
- Understand the bias-variance tradeoff in XGBoost
- Build a production-ready XGBoost model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.preprocessing import StandardScaler
import time

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## XGBoost Hyperparameters: The Complete Guide

### 1. Learning Parameters
- **learning_rate** (eta): Step size shrinkage (0.01-0.3)
- **n_estimators**: Number of boosting rounds (50-1000)

### 2. Tree Parameters
- **max_depth**: Maximum tree depth (3-10)
- **min_child_weight**: Minimum sum of instance weight (1-10)
- **gamma**: Minimum loss reduction for split (0-5)
- **subsample**: Fraction of samples per tree (0.5-1.0)
- **colsample_bytree**: Fraction of features per tree (0.5-1.0)

### 3. Regularization
- **reg_alpha**: L1 regularization (0-1)
- **reg_lambda**: L2 regularization (0-1)

## Load and Prepare Data

In [ ]:
# Load xG data (same as previous notebooks)
matches = sb.matches(competition_id=16, season_id=4)
events = sb.events(match_id=22912)
shots = events[events['type'] == 'Shot'].copy()

def extract_shot_features(shots_df):
    df = shots_df.copy()
    df['distance_to_goal'] = df['location'].apply(
        lambda loc: np.sqrt((120 - loc[0])**2 + (40 - loc[1])**2) if isinstance(loc, list) else None
    )
    def calculate_angle(loc):
        if not isinstance(loc, list):
            return None
        x, y = loc[0], loc[1]
        return np.abs(np.arctan((40 - y) / (120 - x))) * 180 / np.pi
    df['angle_to_goal'] = df['location'].apply(calculate_angle)
    df['right_foot'] = (df['shot_body_part'] == 'Right Foot').astype(int)
    df['open_play'] = (df['shot_type'] == 'Open Play').astype(int)
    df['goal'] = (df['shot_outcome'] == 'Goal').astype(int)
    return df

shots = extract_shot_features(shots)
feature_cols = ['distance_to_goal', 'angle_to_goal', 'right_foot', 'open_play']
X = shots[feature_cols].dropna()
y = shots.loc[X.index, 'goal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data ready: {len(X_train)} training, {len(X_test)} test")

## Grid Search: Systematic Tuning

In [ ]:
# Define parameter grid
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [50, 100, 200],
    'subsample': [0.8, 1.0]
}

# Create base model
base_model = xgb.XGBClassifier(objective='binary:logistic', random_state=42)

# Grid search with cross-validation
print("Starting Grid Search (this may take a while)...")
start_time = time.time()

grid_search = GridSearchCV(
    base_model,
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nGrid Search completed in {time.time() - start_time:.1f} seconds")
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.3f}")
print(f"Test AUC: {roc_auc_score(y_test, grid_search.predict_proba(X_test_scaled)[:, 1]):.3f}")

## Randomized Search: Faster Alternative

In [ ]:
# Define parameter distributions
from scipy.stats import uniform, randint

param_dist = {
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.29),
    'n_estimators': randint(50, 300),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1)
}

print("Starting Randomized Search...")
start_time = time.time()

random_search = RandomizedSearchCV(
    base_model,
    param_dist,
    n_iter=20,  # Number of parameter combinations to try
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train_scaled, y_train)

print(f"\nRandomized Search completed in {time.time() - start_time:.1f} seconds")
print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best CV AUC: {random_search.best_score_:.3f}")
print(f"Test AUC: {roc_auc_score(y_test, random_search.predict_proba(X_test_scaled)[:, 1]):.3f}")

## Early Stopping: Efficient Training

In [ ]:
# Train with early stopping
eval_set = [(X_test_scaled, y_test)]

xgb_early = xgb.XGBClassifier(
    n_estimators=1000,  # Set high, early stopping will find optimal
    max_depth=4,
    learning_rate=0.1,
    objective='binary:logistic',
    random_state=42
)

xgb_early.fit(
    X_train_scaled, y_train,
    eval_set=eval_set,
    eval_metric='auc',
    early_stopping_rounds=10,  # Stop if no improvement for 10 rounds
    verbose=False
)

print(f"Optimal number of trees: {xgb_early.best_iteration}")
print(f"Best validation AUC: {xgb_early.best_score:.3f}")

## Summary

In this notebook, we:

1. ✅ Learned all XGBoost hyperparameters
2. ✅ Used GridSearchCV for systematic tuning
3. ✅ Used RandomizedSearchCV for faster exploration
4. ✅ Implemented early stopping
5. ✅ Built a production-ready model

## Key Takeaways

- **Grid Search**: Exhaustive but slow
- **Randomized Search**: Faster, often finds good solutions
- **Early Stopping**: Prevents overfitting and saves time
- **Start simple**: Tune learning_rate and max_depth first
- **Production**: Use early stopping with a validation set

## Practice Exercises

1. **Bayesian Optimization**: Try hyperopt or optuna for smarter search
2. **Learning Curves**: Plot performance vs n_estimators
3. **Feature Engineering**: Add more features and retune
4. **Ensemble**: Combine multiple tuned models
5. **Production Pipeline**: Create a full preprocessing + tuning pipeline